# 🦀 Day 6: Serialization — serde with TOML, YAML, RON
---

Today we explore **serialization** with serde and multiple formats.

- **serde**: Universal framework; format crates handle encoding/decoding
- **TOML**: Config files (Cargo.toml style), nested tables, arrays
- **YAML**: Human-readable, indentation-based
- **RON**: Rusty Object Notation — Rust-native format
- **JSON**: Ubiquitous; serde_json for compatibility
- **Format comparison**: When to use which

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep toml = "0.8"
:dep serde_yaml = "0.9"
:dep ron = "0.8"
:dep serde_json = "1"

In [ ]:
use serde::{Deserialize, Serialize};

#[derive(Debug, Serialize, Deserialize)]
struct Config {
    name: String,
    version: String,
    enabled: bool,
    tags: Vec<String>,
}

let config = Config {
    name: "my-app".into(),
    version: "1.0.0".into(),
    enabled: true,
    tags: vec!["rust".into(), "cli".into()],
};

// Serialize to all 4 formats
println!("=== JSON ===");
println!("{}", serde_json::to_string_pretty(&config).unwrap());
println!("\n=== TOML ===");
println!("{}", toml::to_string(&config).unwrap());
println!("\n=== YAML ===");
println!("{}", serde_yaml::to_string(&config).unwrap());
println!("\n=== RON ===");
println!("{}", ron::ser::to_string_pretty(&config, ron::ser::PrettyConfig::default()).unwrap());

In [ ]:
// Deserialize from each format
let json_str = r#"{"name":"test","version":"0.1.0","enabled":false,"tags":["a","b"]}"#;
let config: Config = serde_json::from_str(json_str).unwrap();
println!("From JSON: {:?}", config);

let toml_str = r#"
name = "toml-config"
version = "2.0.0"
enabled = true
tags = ["config", "toml"]
"#;
let config: Config = toml::from_str(toml_str).unwrap();
println!("From TOML: {:?}", config);

In [ ]:
// Config parser that auto-detects format from extension
fn parse_config(content: &str, format: &str) -> Result<Config, Box<dyn std::error::Error>> {
    match format {
        "json" => Ok(serde_json::from_str(content)?),
        "toml" => Ok(toml::from_str(content)?),
        "yaml" | "yml" => Ok(serde_yaml::from_str(content)?),
        "ron" => Ok(ron::from_str(content)?),
        _ => Err("Unknown format".into()),
    }
}

let json = r#"{"name":"x","version":"1","enabled":true,"tags":[]}"#;
let cfg = parse_config(json, "json").unwrap();
println!("Parsed: {:?}", cfg);

## Format comparison

| Format | Use case | Pros | Cons |
|--------|----------|------|------|
| **TOML** | Config files | Readable, Cargo standard | Less flexible |
| **YAML** | Config, CI | Human-friendly, comments | Indentation-sensitive |
| **RON** | Rust apps | Rust-native, preserves types | Less tooling |
| **JSON** | APIs, data exchange | Universal, tooling | No comments |

## 📝 Exercise: Config converter

Build a config converter: read from any format (json/toml/yaml/ron), write to any format.
Function signature: `convert(input: &str, from: &str, to: &str) -> Result<String, Error>`

In [ ]:
// YOUR CODE HERE
// fn convert(content: &str, from: &str, to: &str) -> Result<String, Box<dyn std::error::Error>> {
//     let config: Config = parse_config(content, from)?;
//     let out = match to {
//         "json" => serde_json::to_string_pretty(&config)?,
//         "toml" => toml::to_string(&config)?,
//         "yaml" => serde_yaml::to_string(&config)?,
//         "ron" => ron::ser::to_string_pretty(&config, ron::ser::PrettyConfig::default())?,
//         _ => return Err("Unknown output format".into()),
//     };
//     Ok(out)
// }

## 🎯 Key Takeaways

1. **serde** is the universal serialization framework; format crates do the encoding
2. **#[derive(Serialize, Deserialize)]** on structs enables serialization
3. **TOML** for config (Cargo.toml); **YAML** for human config; **RON** for Rust-native
4. Same struct can serialize/deserialize to all formats
5. Auto-detect format from file extension for flexible config loading
6. **serde_json** for JSON; toml, serde_yaml, ron for their formats

---
**Tomorrow:** Mini-project — Personal dev-tools CLI